In [2]:
# this is the import section 

import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
import re
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.base import BaseEstimator,TransformerMixin
from text_cleaner import TextCleaner

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# data loading section

df = pd.read_csv("E:/CustomerS_pred/electronics_small.csv")

C:\Users\ACER\AppData\Local\Temp\ipykernel_5152\704518817.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("E:/CustomerS_pred/electronics_small.csv")


In [4]:
# dropping irrelevant columns and checking for missing values

df.drop(columns = ["reviewTime","summary"],inplace = True)
df = df.dropna(subset="reviewText")
print(df.isna().sum())

overall       0
vote          0
verified      0
reviewText    0
dtype: int64


In [5]:
# creating new target column based on overall column

def label_sentiment(x):
    if x <= 2:
        return 0
    if x >= 4:
        return 1
    else:
        return None
df["sentiment"] = df["overall"].apply(label_sentiment)
df.dropna(subset=["sentiment"],inplace = True)
df["sentiment"] = df["sentiment"].astype(int)

In [6]:
# splitting of the data

X = df[["vote","verified","reviewText"]]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y ,random_state=42)

In [7]:
# using column tramsformer to tfidf and scale the data in 1 go

text_pipeline = Pipeline([("cleaner",TextCleaner()),
                          ("tfidf",TfidfVectorizer())
                         ])
                          
preprocessor = ColumnTransformer(transformers = [('text',text_pipeline,'reviewText'),
                                                 ('num',StandardScaler(),['vote','verified'])
                                                ],remainder = "drop"
                                )

In [8]:
# from here we will build an ml model

model = Pipeline([("preprocessor",preprocessor),
                  ("classifier",LogisticRegression(class_weight='balanced',max_iter=1000))
                   ])
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

In [9]:
import joblib
joblib.dump(model,"hybrid_sentiment_analyzer_model.pkl")

['hybrid_sentiment_analyzer_model.pkl']